# ScamGuard Round 2 (Approach D): Full GPU Fine-Tuning of `xlm-roberta-base` & `multilingual-e5-base`

This self-contained notebook runs full 12-layer fine-tuning on a GPU (Google Colab T4/L4 or Kaggle P100/T4) across **3 random seeds (`42, 100, 2026`)** for:
1. **`FacebookAI/xlm-roberta-base`** (278M parameters, MIT Licence)
2. **`intfloat/multilingual-e5-base`** (278M parameters, MIT Licence, using the required `"query: "` prefix)

---

## Instructions: What to Upload & Where to Put Outputs

### 1. Enable GPU Runtime
- **Google Colab**: `Runtime` $\rightarrow$ `Change runtime type` $\rightarrow$ `Hardware accelerator: T4 GPU`.
- **Kaggle**: `Settings` $\rightarrow$ `Accelerator` $\rightarrow$ `GPU T4 x2` or `GPU P100`.

### 2. Upload These 6 Data Files to `/content/data/` (or Current Working Directory)
From your local ScamGuard repository, upload:
- `ml/data/processed/v1/train_augmented.jsonl`
- `ml/data/processed/v1/val.jsonl`
- `ml/data/processed/v1/test.jsonl`
- `ml/data/processed/v1/gold.jsonl`
- `ml/data/processed/v1/telegram_test.jsonl`
- `ml/eval/hard_negatives_v2.jsonl`

### 3. Where to Place the Output Files Back in the Repo
When training completes, the final cell downloads:
1. `round2_gpu_transformer_results.json` $\rightarrow$ place in **`ml/reports/round2_gpu_transformer_results.json`**
2. `best_xlm_roberta_base.zip` / `best_multilingual_e5_base.zip` $\rightarrow$ extract into **`ml/models/`** (git-ignored)

In [ ]:
!pip install -q transformers accelerate scikit-learn scipy torch

In [ ]:
import json
import os
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from scipy.stats import beta
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

ALL_CLASSES = list(range(1, 14))
SEEDS = [42, 100, 2026]
THRESHOLD = 0.38

In [ ]:
def light_clean(text: str, prefix: str = "") -> str:
    """Light-clean text only: strip URLs and @handles, preserve case, numbers, diacritics, and script."""
    cleaned = re.sub(r"https?://\S+|www\.\S+|t\.me/\S+", " ", text)
    cleaned = re.sub(r"@[A-Za-z0-9_]+", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return f"{prefix}{cleaned}" if prefix else cleaned


def find_data_file(filename: str) -> Path:
    candidates = [
        Path("/content/data") / filename,
        Path("/content") / filename,
        Path(filename),
        Path("../data/processed/v1") / filename,
        Path("../eval") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Please upload it to /content/data/ or current directory.")


def load_jsonl_split(filename: str) -> tuple[list[str], list[int]]:
    path = find_data_file(filename)
    texts, labels = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                d = json.loads(line)
                texts.append(d["text"])
                labels.append(int(d.get("primary_label", d.get("gold_category_id", 10))))
    return texts, labels


def clopper_pearson_ci(k: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    low = 0.0 if k == 0 else float(beta.ppf(alpha / 2.0, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(1.0 - alpha / 2.0, k + 1, n - k))
    return round(low, 4), round(high, 4)


def bootstrap_f1_ci(y_true: list[int], y_pred: list[int], n_boot: int = 1000, seed: int = 42) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(float(f1_score(yt[idx], yp[idx], labels=ALL_CLASSES, average="macro", zero_division=0)))
    scores.sort()
    return round(scores[int(0.025 * n_boot)], 4), round(scores[int(0.975 * n_boot)], 4)

In [ ]:
class NarrativeDataset(Dataset):
    def __init__(self, texts: list[str], labels: list[int], tokenizer, prefix: str = "", max_len: int = 128):
        cleaned = [light_clean(t, prefix=prefix) for t in texts]
        enc = tokenizer(cleaned, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor([lbl - 1 for lbl in labels], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def compute_class_weights(labels: list[int]) -> torch.Tensor:
    counts = Counter(labels)
    total = len(labels)
    weights = []
    for c in ALL_CLASSES:
        cnt = max(1, counts.get(c, 1))
        w = np.sqrt(total / (len(ALL_CLASSES) * cnt))
        weights.append(float(np.clip(w, 0.5, 4.0)))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


@torch.no_grad()
def predict_proba_batched(model, tokenizer, texts: list[str], prefix: str = "", max_len: int = 128, batch_size: int = 64) -> np.ndarray:
    model.eval()
    cleaned = [light_clean(t, prefix=prefix) for t in texts]
    out = []
    for i in range(0, len(cleaned), batch_size):
        batch = cleaned[i : i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        out.append(probs)
    return np.vstack(out)

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_seed(
    model_name: str,
    prefix: str,
    seed: int,
    splits: dict,
    lr: float = 2.5e-5,
    epochs: int = 5,
    batch_size: int = 32,
    patience: int = 2,
) -> dict:
    set_all_seeds(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=13).to(DEVICE)

    train_texts, train_y = splits["train"]
    val_texts, val_y = splits["val"]

    train_ds = NarrativeDataset(train_texts, train_y, tokenizer, prefix=prefix)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    class_weights = compute_class_weights(train_y)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val_f1 = -1.0
    best_state = None
    no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ep_loss += float(loss.item())

        val_probs = predict_proba_batched(model, tokenizer, val_texts, prefix=prefix)
        val_preds = [ALL_CLASSES[i] if val_probs[idx].max() >= THRESHOLD else 0 for idx, i in enumerate(val_probs.argmax(axis=1))]
        val_f1 = float(f1_score(val_y, val_preds, labels=ALL_CLASSES, average="macro", zero_division=0))
        print(f"  [{model_name} | seed={seed}] Epoch {ep}/{epochs} | loss={ep_loss/len(train_loader):.4f} | val_macro_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping triggered at epoch {ep}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    # Evaluate on held-out splits
    eval_out = {"seed": seed, "best_val_macro_f1": round(best_val_f1, 4), "splits": {}}
    for s_name in ["test", "gold_150", "gold_unique_30", "telegram_test"]:
        s_texts, s_y = splits[s_name]
        probs = predict_proba_batched(model, tokenizer, s_texts, prefix=prefix)
        preds = [ALL_CLASSES[i] if probs[idx].max() >= THRESHOLD else 0 for idx, i in enumerate(probs.argmax(axis=1))]
        mf1 = float(f1_score(s_y, preds, labels=ALL_CLASSES, average="macro", zero_division=0))
        ci_l, ci_h = bootstrap_f1_ci(s_y, preds)
        acc = float(accuracy_score(s_y, preds))
        maj_c, maj_cnt = Counter(s_y).most_common(1)[0]
        eval_out["splits"][s_name] = {
            "macro_f1": round(mf1, 4),
            "ci_95": [ci_l, ci_h],
            "accuracy": round(acc, 4),
            "majority_baseline": round(maj_cnt / len(s_y), 4),
        }

    # Hard negatives v2 FPR + Clopper-Pearson 95% CI
    hn_texts, _ = splits["hard_negatives_v2"]
    hn_probs = predict_proba_batched(model, tokenizer, hn_texts, prefix=prefix)
    hn_preds = [ALL_CLASSES[i] if hn_probs[idx].max() >= THRESHOLD else 0 for idx, i in enumerate(hn_probs.argmax(axis=1))]
    fp = sum(1 for p in hn_preds if p not in (0, 10))
    cp_l, cp_h = clopper_pearson_ci(fp, len(hn_texts))
    eval_out["hard_negatives_v2"] = {
        "fp": fp,
        "n": len(hn_texts),
        "fpr": round(fp / len(hn_texts), 4),
        "clopper_pearson_95_ci": [cp_l, cp_h],
    }
    return eval_out

In [ ]:
train_texts, train_y = load_jsonl_split("train_augmented.jsonl")
val_texts, val_y = load_jsonl_split("val.jsonl")
test_texts, test_y = load_jsonl_split("test.jsonl")
gold_texts, gold_y = load_jsonl_split("gold.jsonl")
tg_texts, tg_y = load_jsonl_split("telegram_test.jsonl")
hn_v2_texts, hn_v2_y = load_jsonl_split("hard_negatives_v2.jsonl")

splits_data = {
    "train": (train_texts, train_y),
    "val": (val_texts, val_y),
    "test": (test_texts, test_y),
    "gold_150": (gold_texts, gold_y),
    "gold_unique_30": (gold_texts[:30], gold_y[:30]),
    "telegram_test": (tg_texts, tg_y),
    "hard_negatives_v2": (hn_v2_texts, hn_v2_y),
}

CANDIDATES = [
    ("FacebookAI/xlm-roberta-base", ""),
    ("intfloat/multilingual-e5-base", "query: "),
]

all_gpu_results = {}
for model_id, prefix in CANDIDATES:
    print("=" * 80)
    print(f"Running 3-seed full GPU fine-tuning for {model_id} (prefix={prefix!r})")
    print("=" * 80)
    seed_runs = []
    for s in SEEDS:
        res = train_one_seed(model_id, prefix, s, splits_data)
        seed_runs.append(res)

    summary = {"runs": seed_runs, "aggregate_mean_sd": {}}
    for s_name in ["test", "gold_150", "gold_unique_30", "telegram_test"]:
        vals = [r["splits"][s_name]["macro_f1"] for r in seed_runs]
        summary["aggregate_mean_sd"][s_name] = {
            "mean_macro_f1": round(float(np.mean(vals)), 4),
            "sd_macro_f1": round(float(np.std(vals, ddof=1)), 4),
        }
    fprs = [r["hard_negatives_v2"]["fpr"] for r in seed_runs]
    summary["aggregate_mean_sd"]["hard_negatives_v2_fpr"] = {
        "mean_fpr": round(float(np.mean(fprs)), 4),
        "sd_fpr": round(float(np.std(fprs, ddof=1)), 4),
    }
    all_gpu_results[model_id] = summary

with open("round2_gpu_transformer_results.json", "w", encoding="utf-8") as f:
    json.dump(all_gpu_results, f, indent=2)
print("Saved round2_gpu_transformer_results.json")